In [2]:
import os
import numpy as np
import mne
from mne.preprocessing import ICA

In [3]:
BASE_DIR = r"C:\Users\enneh\ds002778"
OUTPUT_DIR = r"C:\Users\enneh\eeg_band_data-hc"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SFREQ = 512
DURATION = 180      # seconds (3 minutes)
EPOCH_LEN = 1.0     # 1 second

FREQ_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (13, 30),
    "gamma": (30, 48)
}

DROP_CHANNELS = [
    'EXG1','EXG2','EXG3','EXG4',
    'EXG5','EXG6','EXG7','EXG8','Status'
]

In [4]:
def load_raw(file_path):
    raw = mne.io.read_raw_bdf(file_path, preload=True, verbose=False)
    raw.crop(tmax=DURATION)
    raw.drop_channels(DROP_CHANNELS)
    return raw

In [5]:
def apply_montage(raw):
    montage = mne.channels.make_standard_montage('biosemi32')
    raw.set_montage(montage, on_missing='warn')
    return raw


In [6]:
def bandpass_raw(raw, l_freq, h_freq):
    return raw.copy().filter(
        l_freq=l_freq,
        h_freq=h_freq,
        verbose=False
    )

In [7]:
def apply_ica(raw):
    ica = ICA(n_components=32, random_state=97, max_iter="auto")
    ica.fit(raw)

    eog_idx, _ = ica.find_bads_eog(
        raw, ch_name=['Fp2', 'F8'], threshold=1.96
    )

    ica.exclude = eog_idx
    raw_clean = ica.apply(raw)
    return raw_clean

In [8]:
def epoch_raw(raw):
    events = mne.make_fixed_length_events(raw, duration=EPOCH_LEN)
    epochs = mne.Epochs(
        raw,
        events,
        tmin=0,
        tmax=EPOCH_LEN,
        baseline=None,
        preload=True,
        verbose=False
    )
    return epochs.get_data()[:, :, :512]

In [10]:
def process_subject(subject_id):
    file_path = os.path.join(
        BASE_DIR,
        subject_id,
        "ses-hc",
        "eeg",
        f"{subject_id}_ses-hc_task-rest_eeg.bdf"
    )

    print(f"Processing {subject_id}")

    raw = load_raw(file_path)
    raw = apply_montage(raw)
    raw = raw.filter(0.5, 50, verbose=False)   # broadband filter
    raw = apply_ica(raw)

    band_data = {}

    for band, (low, high) in FREQ_BANDS.items():
        raw_band = bandpass_raw(raw, low, high)
        band_epochs = epoch_raw(raw_band)
        band_data[band] = band_epochs

    # Stack bands → CNN format
    X = np.stack([
        band_data['delta'],
        band_data['theta'],
        band_data['alpha'],
        band_data['beta'],
        band_data['gamma']
    ], axis=1)

    print(f"{subject_id} shape: {X.shape}")

    # Save
    np.save(os.path.join(OUTPUT_DIR, f"{subject_id}_X.npy"), X)

In [11]:
PD_SUBJECTS = [
    "sub-hc1","sub-hc10","sub-hc18","sub-hc2",
    "sub-hc20","sub-hc21","sub-hc24","sub-hc25",
    "sub-hc29","sub-hc30","sub-hc32","sub-hc33",
    "sub-hc4","sub-hc7","sub-hc8","sub-hc31"
]

for subject in PD_SUBJECTS:
    process_subject(subject)

Processing sub-hc1
Fitting ICA to data using 32 channels (please be patient, this may take a while)
Selecting by number: 32 components
Fitting ICA took 14.5s.
Using EOG channels: Fp2, F8
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transitio

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\decomposition\_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 sam

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\decomposition\_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 sam

...

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 sam

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\decomposition\_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 sam

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\decomposition\_fastica.py:128: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 samples (10.000 s)

... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 5120 sam